# 02 - Reports (SQL + PL/SQL)
These are the "Reporting & Analytics" from our proposal, written as real SQL
run directly against Oracle - you can show this notebook as project evidence.

In [ ]:
import os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'internship_system.settings')
import django
django.setup()
from django.db import connection

def run_query(sql):
    '''Run a SELECT and print the results as a simple table.'''
    with connection.cursor() as cursor:
        cursor.execute(sql)
        columns = [d[0] for d in cursor.description]
        print(' | '.join(columns))
        print('-' * 60)
        for row in cursor.fetchall():
            print(' | '.join(str(v) for v in row))

## Report 1: Applications received per company

In [ ]:
run_query('''
    SELECT c.name AS company, COUNT(*) AS total_applications
    FROM application a
    JOIN internship i ON a.internship_id = i.id
    JOIN company c    ON i.company_id = c.id
    GROUP BY c.name
    ORDER BY total_applications DESC
''')

## Report 2: Most demanded skills

In [ ]:
run_query('''
    SELECT s.name AS skill, COUNT(*) AS times_demanded
    FROM internship_skill x
    JOIN skill s ON x.skill_id = s.id
    GROUP BY s.name
    ORDER BY times_demanded DESC
''')

## Report 3: Placement statistics

In [ ]:
run_query('''
    SELECT status, COUNT(*) AS total, ROUND(AVG(stipend), 0) AS avg_stipend
    FROM placement p
    JOIN offer o ON p.offer_id = o.id
    GROUP BY status
''')

## Report 4: Top students by number of applications

In [ ]:
run_query('''
    SELECT st.roll_no, st.name, COUNT(*) AS applications
    FROM application a
    JOIN student st ON a.student_id = st.id
    GROUP BY st.roll_no, st.name
    ORDER BY applications DESC
    FETCH FIRST 5 ROWS ONLY
''')

## PL/SQL: a stored FUNCTION in Oracle
A PL/SQL function is a small program stored *inside the database*.
We create it once, then call it like a built-in function. This is exactly the
"PL/SQL" part mentioned in our proposal.

In [ ]:
with connection.cursor() as cursor:
    cursor.execute('''
        CREATE OR REPLACE FUNCTION count_students RETURN NUMBER IS
            total NUMBER;
        BEGIN
            SELECT COUNT(*) INTO total FROM student;
            RETURN total;
        END;
    ''')
print('Function created.')

In [ ]:
run_query("SELECT count_students() AS total_students FROM dual")

## Same report using Django ORM (Python way) - for comparison

In [ ]:
from portal.models import Application
from django.db.models import Count

rows = (Application.objects.values('internship__company__name')
        .annotate(total=Count('id'))
        .order_by('-total'))
for r in rows:
    print(r['internship__company__name'], '->', r['total'])